# PDF → sayfa başına WebP

Drive'daki bir klasördeki her PDF için aynı klasörde **PDF adıyla bir alt klasör** açar ve her sayfayı
`page_1.webp`, `page_2.webp` ... olarak oraya yazar (ad `PAGE_NAME` ile değişir; varsayılan, çalışma sayfası
editörünün toplu yüklemede beklediği `<kitap>/page_<n>.webp` düzenidir).

- Klasöre Drive API ile ID üzerinden erişir: klasör "Benimle paylaşılanlar"da veya paylaşılan Drive'da olsa da çalışır.
- **Kaldığı yerden devam eder:** Colab koparsa hücreyi yeniden çalıştır; bitmiş PDF'ler indirilmeden atlanır,
  yarım kalanın yalnız eksik sayfaları üretilir.
- Alt klasörlerdeki PDF'lere bakmaz (yalnız verilen klasörün kendisi).

Varsayılan en net çıktıdır: sayfa başına otomatik çözünürlük + kayıpsız WebP (ayarlar 3. hücrede).

Hücreleri sırayla çalıştır; ikinci hücrede Google hesabı izni istenir. GPU gerekmez.

In [ ]:
!pip -q install pymupdf

In [ ]:
from google.colab import auth
from googleapiclient.discovery import build

auth.authenticate_user()
svc = build('drive', 'v3', cache_discovery=False)

In [ ]:
# Klasör linki veya ID'si
FOLDER = ''
# Netlik: 'auto' her sayfaya ayrı çözünürlük seçer. Taranmış sayfada gömülü taramanın kendi çözünürlüğü
# (en az MIN_DPI, en çok MAX_DPI; taramanın üstüne çıkmak netlik katmaz, yalnız dosyayı büyütür),
# metni seçilebilen (vektör) sayfada VECTOR_DPI. Sabit istersen sayı yaz (örn. DPI = 300).
DPI = 'auto'
VECTOR_DPI = 600     # A4'te 4961×7016 px (en net); iPad/telefonda tuval yavaşsa 400
MIN_DPI = 300
MAX_DPI = 600
LOSSLESS = True      # kayıpsız WebP: sıkıştırma kaybı yok (kitap sayfasında ~0.6-0.8 MB/sayfa)
WEBP_QUALITY = 95    # yalnız LOSSLESS = False iken (kayıplı; renkleri yarım çözünürlükte saklar)
DRY_RUN = False      # True: yalnız ne yapılacağını yazar, dosya üretmez
# Sayfa dosya adı ({n}: sayfa no, {pdf}: PDF adı). 'page_{n}' uygulamadaki çalışma sayfası editörünün
# toplu yüklemede beklediği ad (<kitap>/page_<n>.webp). Drive'da sıralı görünsün istersen: '{pdf}-{n:03d}'
PAGE_NAME = 'page_{n}'

In [ ]:
import io
import os
import re
import traceback

import pymupdf
from PIL import Image
from googleapiclient.http import MediaIoBaseDownload, MediaIoBaseUpload

FOLDER_MIME = 'application/vnd.google-apps.folder'
SHORTCUT_MIME = 'application/vnd.google-apps.shortcut'
PDF_MIME = 'application/pdf'
MAX_SIDE = 8000  # px; kitap sayfası için fazlasıyla yeter, aşırı büyük sayfada belleği korur
RETRIES = 5
TMP = '/content/pdf_tmp'


def folder_id_of(value):
    m = re.search(r'folders/([\w-]+)', value)
    return m.group(1) if m else value.strip()


def q_escape(s):
    return s.replace('\\', '\\\\').replace("'", "\\'")


def list_children(parent_id, extra_q=''):
    q = f"'{parent_id}' in parents and trashed = false" + (f' and {extra_q}' if extra_q else '')
    items, token = [], None
    while True:
        r = svc.files().list(
            q=q, pageToken=token, pageSize=1000,
            fields='nextPageToken, files(id, name, mimeType, appProperties, shortcutDetails)',
            supportsAllDrives=True, includeItemsFromAllDrives=True,
        ).execute(num_retries=RETRIES)
        items += r.get('files', [])
        token = r.get('nextPageToken')
        if not token:
            return items


def find_folder(parent_id, name):
    found = list_children(parent_id, f"mimeType = '{FOLDER_MIME}' and name = '{q_escape(name)}'")
    return found[0] if found else None


def create_folder(parent_id, name):
    return svc.files().create(
        body={'name': name, 'mimeType': FOLDER_MIME, 'parents': [parent_id]},
        fields='id, name, appProperties', supportsAllDrives=True,
    ).execute(num_retries=RETRIES)


def settings_key():
    # Netlik ayarı değişince bitmiş PDF'ler atlanmaz; yalnız eksik sayfa adları üretilir (var olanlar yeniden yazılmaz).
    q = 'lossless' if LOSSLESS else f'q{WEBP_QUALITY}'
    return f'{DPI}/{VECTOR_DPI}/{MIN_DPI}-{MAX_DPI}/{q}' if DPI == 'auto' else f'{DPI}/{q}'


def mark_done(folder_id, source_id, pages):
    # Bitti işareti klasörün görünmez appProperties'inde; tekrar çalıştırmada PDF indirilmeden atlanır.
    svc.files().update(
        fileId=folder_id, supportsAllDrives=True,
        body={'appProperties': {'pdf2webp_source': source_id, 'pdf2webp_pages': str(pages),
                                'pdf2webp_dpi': settings_key()}},
    ).execute(num_retries=RETRIES)


def is_done(folder, source_id):
    props = folder.get('appProperties') or {}
    return props.get('pdf2webp_source') == source_id and props.get('pdf2webp_dpi') == settings_key()


def download(file_id, path):
    req = svc.files().get_media(fileId=file_id, supportsAllDrives=True)
    with open(path, 'wb') as f:
        dl = MediaIoBaseDownload(f, req, chunksize=64 * 1024 * 1024)
        done = False
        while not done:
            _, done = dl.next_chunk(num_retries=RETRIES)


def upload(parent_id, name, data, mime):
    media = MediaIoBaseUpload(io.BytesIO(data), mimetype=mime, resumable=False)
    svc.files().create(
        body={'name': name, 'parents': [parent_id]}, media_body=media,
        fields='id', supportsAllDrives=True,
    ).execute(num_retries=RETRIES)


def page_dpi(page):
    if DPI != 'auto':
        return float(DPI)
    scan = 0.0
    for info in page.get_image_info():
        x0, y0, x1, y1 = info['bbox']
        w_in, h_in = (x1 - x0) / 72, (y1 - y0) / 72
        if w_in < 2 or h_in < 2:  # logo/ikon gibi küçük görseller sayfanın çözünürlüğünü belirlemez
            continue
        scan = max(scan, info['width'] / w_in, info['height'] / h_in)
    if not scan:
        return float(VECTOR_DPI)
    return min(max(scan, MIN_DPI), MAX_DPI)


def render_webp(page):
    zoom = page_dpi(page) / 72
    longest = max(page.rect.width, page.rect.height) * zoom
    if longest > MAX_SIDE:
        zoom *= MAX_SIDE / longest
    pix = page.get_pixmap(matrix=pymupdf.Matrix(zoom, zoom), alpha=False, colorspace=pymupdf.csRGB)
    img = Image.frombytes('RGB', (pix.width, pix.height), pix.samples)
    buf = io.BytesIO()
    if LOSSLESS:
        img.save(buf, 'WEBP', lossless=True, quality=80, method=4)  # quality burada sıkıştırma emeği
    else:
        img.save(buf, 'WEBP', quality=WEBP_QUALITY, method=4)
    return buf.getvalue()


def pdf_sources(folder_id):
    """(drive id, ad) listesi; PDF kısayolları hedef dosyaya çözülür."""
    out = []
    for f in list_children(folder_id):
        if f['mimeType'] == PDF_MIME:
            out.append((f['id'], f['name']))
        elif f['mimeType'] == SHORTCUT_MIME and (f.get('shortcutDetails') or {}).get('targetMimeType') == PDF_MIME:
            out.append((f['shortcutDetails']['targetId'], f['name']))
    return sorted(out, key=lambda x: x[1].lower())


def out_names(sources):
    """PDF adından klasör adı; aynı adlı iki PDF çakışmasın diye ikincisine ID eki."""
    names, used = {}, set()
    for fid, name in sources:
        base = re.sub(r'\.pdf$', '', name, flags=re.IGNORECASE).strip() or fid
        if base.lower() in used:
            base = f'{base}_{fid[:6]}'
        used.add(base.lower())
        names[fid] = base
    return names


def convert_one(parent_id, fid, base):
    folder = find_folder(parent_id, base)
    if folder and is_done(folder, fid):
        print('  atlandı (daha önce bitmiş)')
        return 'skipped', 0
    existing = {c['name'] for c in list_children(folder['id'])} if folder else set()

    os.makedirs(TMP, exist_ok=True)
    local = os.path.join(TMP, f'{fid}.pdf')
    download(fid, local)
    try:
        # Klasör PDF açılabildikten sonra açılır: bozuk/şifreli PDF boş klasör bırakmaz.
        with pymupdf.open(local) as doc:
            if doc.needs_pass:
                print('  ŞİFRELİ PDF, atlandı')
                return 'failed', 0
            n = doc.page_count
            names = [PAGE_NAME.format(n=i + 1, pdf=base) + '.webp' for i in range(n)]
            todo = [i for i in range(n) if names[i] not in existing]
            if DRY_RUN:
                print(f'  {n} sayfa, {len(todo)} webp üretilecek (deneme)')
                return 'done', len(todo)
            if folder is None:
                folder = create_folder(parent_id, base)
            for k, i in enumerate(todo, 1):
                upload(folder['id'], names[i], render_webp(doc[i]), 'image/webp')
                if k % 10 == 0:
                    print(f'  {k}/{len(todo)}')
        mark_done(folder['id'], fid, n)
        print(f'  {n} sayfa, {len(todo)} yeni webp')
        return 'done', len(todo)
    finally:
        os.remove(local)


def run():
    parent_id = folder_id_of(FOLDER)
    if not parent_id:
        raise SystemExit('FOLDER boş: klasör linkini veya ID’sini yaz.')
    sources = pdf_sources(parent_id)
    names = out_names(sources)
    print(f'{len(sources)} PDF bulundu')
    summary = {'done': 0, 'skipped': 0, 'failed': 0, 'pages': 0}
    failed = []
    for k, (fid, name) in enumerate(sources, 1):
        print(f'[{k}/{len(sources)}] {name} -> {names[fid]}/')
        try:
            status, made = convert_one(parent_id, fid, names[fid])
        except Exception:
            traceback.print_exc()
            status, made = 'failed', 0
        summary[status] += 1
        summary['pages'] += made
        if status == 'failed':
            failed.append(name)
    print(f"\nBitti: {summary['done']} işlendi, {summary['skipped']} atlandı, "
          f"{summary['failed']} hata, {summary['pages']} yeni webp")
    if failed:
        print('Hatalı:', *failed, sep='\n  ')


run()